# Connectivity vs Time Pressure across all four case neighbourhoods

This experiment repeats Experiment 2.2 across four Amsterdam neighbourhoods that differ in third place accessibility and transport accessibility, forming a 2×2 typology. The four neighbourhoods are: High/High (dense city centre), High Third Place/Low Transport (inner ring), Low Third Place/High Transport (Zuidoost), and Low/Low (outer Nieuw-West).

The spatial context for each neighbourhood is encoded in three input files: road nodes, road edges, and third place locations, extracted from Amsterdam's buurt shapefile and OSM road network 

 These files are loaded manually into the NetLogo model before each neighbourhood's run by updating the file paths directly in the model — this step is not visible in the code below but is what distinguishes the four experiments. 

In [22]:
import pynetlogo
import pandas as pd
import itertools
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# 1. Explicitly point to the libjvm.dylib file inside NetLogo's application package
# Path on GIS DESKTOP
jvm_path = "C:/Program Files/NetLogo 6.4.0/runtime/bin/server/jvm.dll"
netlogo_home = "C:/Program Files/NetLogo 6.4.0"

#Path on Mac Laptop
# jvm_path = "/Applications/NetLogo 6.4.0/runtime/Contents/Home/lib/server/libjvm.dylib"
# netlogo_home="/Applications/NetLogo 6.4.0"

netlogo = pynetlogo.NetLogoLink(
    gui=False,
    netlogo_home=netlogo_home,
    jvm_path=jvm_path 
)

# Path in GIS DESKTOP
model_path = "C:/Users/15177459/Documents/Bachelor-Thesis/Netlogo/third_place_attributes.nlogo"

# Path in Mac Laptop
# model_path = "/Users/tonyvo/Desktop/Thesis/Bachelor-Thesis/Netlogo/third_place_attributes.nlogo"

netlogo.load_model(model_path)
netlogo.command("setup")
netlogo.report("count places")

3.0

# First Heatmap: High Transport, High Third Place

In [6]:
# Attribute based choice ON
parameter_grid = {
    "number-of-people": [200],
    "baseline-stop-probability": [10, 20, 30, 40],
    "third-place-search-radius": [1, 2, 5, 10],
    "route-third-place-sample-size": [50],
    "rigid-dwell-time": [30],
    "medium-dwell-time": [45],
    "flexible-dwell-time": [60],
    "encounter-odds-increment": [0.10],
    "encounter-weight": [5],
    "social-feedback?": [True],
    "attribute-based-choice?": [True]
}

reporters = [
    "visits-per-person",
    "average-stop-probability",
    "average-social-encounters",
    "average-social-odds-ratio",
    "total-co-presence",
    "co-presence-per-visit",
    "visited-places-count",
    "places-with-co-presence",
    "visit-concentration",
    "average-visited-place-affordability",
    "average-visited-place-welcomingness"
]

results = []

keys = list(parameter_grid.keys())
values = list(parameter_grid.values())

run_id = 0

for combination in itertools.product(*values):
    params = dict(zip(keys, combination))
    
    for seed in range(1, 21):  # 20 repetitions
        run_id += 1
        
        netlogo.command(f"random-seed {seed}")
        
        for parameter, value in params.items():
            if isinstance(value, bool):
                value = "true" if value else "false"
            netlogo.command(f"set {parameter} {value}")
        
        netlogo.command("setup")
        netlogo.command("repeat 1000 [ go ]")
        
        row = {
            "experiment": "attribute_test",
            "run_id": run_id,
            "seed": seed,
            **params,
        }
        
        for reporter in reporters:
            row[reporter] = netlogo.report(reporter)
        
        results.append(row)

exphi_hi = pd.DataFrame(results)

print(exphi_hi.shape)
exphi_hi.head()

(320, 25)


,experiment,run_id,seed,number-of-people,baseline-stop-probability,third-place-search-radius,route-third-place-sample-size,rigid-dwell-time,medium-dwell-time,flexible-dwell-time,...,average-stop-probability,average-social-encounters,average-social-odds-ratio,total-co-presence,co-presence-per-visit,visited-places-count,places-with-co-presence,visit-concentration,average-visited-place-affordability,average-visited-place-welcomingness
0,attribute_test,1,1,200,10,1,50,30,45,60,...,8.983445,3.25,1.2560,2247.0,13.140351,36.0,15.0,0.122807,0.592335,0.677218
1,attribute_test,2,2,200,10,1,50,30,45,60,...,8.564894,2.45,1.2190,1832.0,11.819355,39.0,14.0,0.083871,0.619892,0.654527
2,attribute_test,3,3,200,10,1,50,30,45,60,...,8.994830,2.55,1.2495,2276.0,14.875817,34.0,13.0,0.098039,0.650876,0.605243
3,attribute_test,4,4,200,10,1,50,30,45,60,...,8.667442,2.60,1.2270,2281.0,13.658683,33.0,16.0,0.083832,0.624168,0.615319
4,attribute_test,5,5,200,10,1,50,30,45,60,...,8.651313,2.90,1.2285,2337.0,13.994012,34.0,14.0,0.107784,0.546683,0.588554


In [7]:
grouped_hi_hi = exphi_hi.groupby(["baseline-stop-probability", "third-place-search-radius"]).agg(
    co_presence_mean = ("total-co-presence", "mean"),
    co_presence_std = ("total-co-presence", "std"),
    visits_mean = ("visits-per-person", "mean"),
    encounter_mean = ("average-social-encounters", "mean"),
    concentration_mean = ("visit-concentration", "mean"),
    copresence_per_visit_mean = ("co-presence-per-visit", "mean")
).reset_index()

grouped_hi_hi

,baseline-stop-probability,third-place-search-radius,co_presence_mean,co_presence_std,visits_mean,encounter_mean,concentration_mean,copresence_per_visit_mean
0,10,1,1797.75,423.285132,0.76350,2.3275,0.095442,11.690120
1,10,2,2887.70,754.079229,0.75750,4.4250,0.149964,18.876633
2,10,5,3931.20,898.218154,0.70625,7.2150,0.217125,27.541046
3,10,10,3916.05,871.859143,0.70500,7.1425,0.216182,27.514576
4,20,1,6032.45,769.370110,1.53950,9.6675,0.089077,19.555364
5,20,2,8725.20,1182.050475,1.47050,18.4800,0.153584,29.618426
6,20,5,10221.85,1231.702941,1.36875,28.7825,0.208248,37.262307
7,20,10,10303.50,1445.408429,1.34850,29.0250,0.218150,38.053014
8,30,1,10807.85,1488.441901,2.20900,20.2475,0.089104,24.398507
9,30,2,13600.05,1220.242188,2.06425,36.1950,0.149977,32.913811


# Second Heatmap: High Transport Low Third Place

In [ ]:
# Attribute based choice ON
parameter_grid = {
    "number-of-people": [200],
    "baseline-stop-probability": [10, 20, 30, 40],
    "third-place-search-radius": [1, 2, 5, 10],
    "route-third-place-sample-size": [50],
    "rigid-dwell-time": [30],
    "medium-dwell-time": [45],
    "flexible-dwell-time": [60],
    "encounter-odds-increment": [0.10],
    "encounter-weight": [5],
    "social-feedback?": [True],
    "attribute-based-choice?": [True]
}

reporters = [
    "visits-per-person",
    "average-stop-probability",
    "average-social-encounters",
    "average-social-odds-ratio",
    "total-co-presence",
    "co-presence-per-visit",
    "visited-places-count",
    "places-with-co-presence",
    "visit-concentration",
    "average-visited-place-affordability",
    "average-visited-place-welcomingness"
]

results = []

keys = list(parameter_grid.keys())
values = list(parameter_grid.values())

run_id = 0

for combination in itertools.product(*values):
    params = dict(zip(keys, combination))
    
    for seed in range(1, 21):  # 20 repetitions
        run_id += 1
        
        netlogo.command(f"random-seed {seed}")
        
        for parameter, value in params.items():
            if isinstance(value, bool):
                value = "true" if value else "false"
            netlogo.command(f"set {parameter} {value}")
        
        netlogo.command("setup")
        netlogo.command("repeat 1000 [ go ]")
        
        row = {
            "experiment": "attribute_test",
            "run_id": run_id,
            "seed": seed,
            **params,
        }
        
        for reporter in reporters:
            row[reporter] = netlogo.report(reporter)
        
        results.append(row)

exphi_low = pd.DataFrame(results)

print(exphi_hi.shape)
exphi_low.head()